# 02_内容构建 (Content Builder Agent)

**来源:** [LangChain Docs — Build a content builder agent](https://docs.langchain.com/deep-agents/content-builder)

本教程演示如何使用 Deep Agents 构建一个内容写作智能体。该智能体具有品牌记忆、技能系统、子智能体委派和图片生成能力，能够：加载品牌语调和写作规则、委派 Web 研究给子智能体、按技能模板撰写博客或社交媒体内容、使用 Gemini 生成封面和社交配图。

## 1. 核心概念

| 概念 | 说明 |
|------|------|
| 长期记忆 (Memory) | 通过 `AGENTS.md` 文件定义品牌语调和写作标准 |
| 技能 (Skills) | `skills/` 目录中的 `SKILL.md` 文件，定义特定内容类型的写作流程 |
| 子智能体 | 用于搜索研究的专用子智能体 |
| 文件系统后端 | 用于读写帖子、研究笔记和图片 |
| 自定义工具 | Web 搜索和 AI 图片生成 |

## 2. 环境准备

安装依赖并设置 API 密钥。

In [ ]:
# 安装核心依赖
# pip install deepagents google-genai pillow pyyaml rich tavily-python langchain

# 设置 API 密钥
import os

# os.environ["ANTHROPIC_API_KEY"] = "your-api-key"  # 主模型
# os.environ["GOOGLE_API_KEY"] = "your-api-key"    # Gemini 图片生成
# os.environ["TAVILY_API_KEY"] = "your-api-key"    # Web 搜索
# os.environ["LANGSMITH_API_KEY"] = "your-api-key" # 可选：追踪

print("环境准备完成")

## 3. 品牌记忆文件 (AGENTS.md)

`AGENTS.md` 文件定义了品牌的语调、写作标准和内容支柱。当智能体创建时，该文件被加载到系统提示中。

In [ ]:
# AGENTS.md 内容示例 - 品牌记忆配置
AGENTS_MD_CONTENT = """# Content Writer Agent

You are a content writer for a technology company.

## Brand Voice
- **Professional but approachable**: Write like a knowledgeable colleague
- **Clear and direct**: Avoid jargon unless necessary
- **Confident but not arrogant**: Share expertise without being condescending
- **Engaging**: Use concrete examples, analogies, and stories

## Writing Standards
1. **Use active voice**: "The agent processes requests" not "Requests are processed"
2. **Lead with value**: Start with what matters to the reader
3. **One idea per paragraph**: Keep paragraphs focused and scannable
4. **Concrete over abstract**: Use specific examples and numbers
5. **End with action**: Every piece should leave the reader knowing what to do next

## Content Pillars
- AI agents and automation
- Developer tools and productivity
- Software architecture and best practices
- Emerging technologies and trends

## Formatting Guidelines
- Use headers (H2, H3) to break up content
- Include code examples where relevant
- Add bullet points for lists of 3+ items
- Keep sentences under 25 words when possible

## Research Requirements
Before writing:
1. Use the `researcher` subagent for in-depth topic research
2. Gather at least 3 credible sources
3. Identify the key points readers need to understand
"""

# 将品牌记忆写入文件
import os
from pathlib import Path

project_dir = Path("content-builder-agent")
project_dir.mkdir(exist_ok=True)

with open(project_dir / "AGENTS.md", "w") as f:
    f.write(AGENTS_MD_CONTENT)

print(f"AGENTS.md 已写入 {project_dir / 'AGENTS.md'}")

## 4. 子智能体配置 (subagents.yaml)

   定义研究子智能体，配置其使用的模型、工具和系统提示。

In [ ]:
import yaml

# 子智能体配置
SUBAGENTS_CONFIG = {
    "researcher": {
        "description": (
            "ALWAYS use this first to research any topic before writing content. "
            "Searches the web for current information, statistics, and sources."
        ),
        "model": "anthropic:claude-haiku-4-5-20251001",
        "system_prompt": (
            "You are a research assistant. You have access to web_search and write_file tools.\n\n"
            "## Your Process\n"
            "1. Use web_search to find information on the topic\n"
            "2. Make 2-3 targeted searches with specific queries\n"
            "3. Gather key statistics, quotes, and examples\n"
            "4. Save findings to the file path specified in your task\n\n"
            "## Important\n"
            "- Always include source URLs in your findings\n"
            "- Keep findings concise but informative"
        ),
        "tools": ["web_search"]
    }
}

# 写入 YAML 文件
with open(project_dir / "subagents.yaml", "w") as f:
    yaml.dump(SUBAGENTS_CONFIG, f, default_flow_style=False, allow_unicode=True)

print(f"subagents.yaml 已写入 {project_dir / 'subagents.yaml'}")

## 5. 技能文件夹 (Skills)

   技能文件夹包含特定内容类型的写作模板。每个技能是一个文件夹中的 `SKILL.md` 文件，包含 YAML 前置元数据和写作指令。

   ### 博客写作技能 (blog-post)

   定义博客文章的写作流程、SEO 优化指南和封面图片生成标准。

In [ ]:
# 创建博客写作技能文件夹
skills_blog = project_dir / "skills" / "blog-post"
skills_blog.mkdir(parents=True, exist_ok=True)

SKILL_BLOG_POST = """---
name: blog-post
description: Writes and structures long-form blog posts, creates tutorial outlines, and optimizes content for SEO with cover image generation.
---

# Blog Post Writing Skill

## Research First (Required)
- Use the `task` tool with `subagent_type: "researcher"`
- Specify BOTH the topic AND where to save findings
- After research completes, read the findings file before writing

## Output Structure
- `blogs/<slug>/post.md` - Blog post content
- `blogs/<slug>/hero.png` - Cover image (REQUIRED)

## Blog Post Structure
1. Hook (Opening) - Compelling question or statement
2. Context (The Problem) - Why this matters
3. Main Content (The Solution) - 3-5 main sections
4. Practical Application - Step-by-step instructions
5. Conclusion & CTA - Key takeaways and call-to-action

## SEO Considerations
- Main keyword in title and first paragraph
- Keyword used naturally 3-5 times
- Title under 60 characters
- Meta description 150-160 characters

## Cover Image Generation
- Use `generate_cover` tool after writing
- Prompt should include: subject, style, composition, color palette, lighting
"""

with open(skills_blog / "SKILL.md", "w") as f:
    f.write(SKILL_BLOG_POST)

print(f"博客技能已创建: {skills_blog / 'SKILL.md'}")

In [ ]:
# 创建社交媒体写作技能文件夹
skills_social = project_dir / "skills" / "social-media"
skills_social.mkdir(parents=True, exist_ok=True)

SKILL_SOCIAL_MEDIA = """---
name: social-media
description: Drafts engaging social media posts, writes hooks, suggests hashtags, creates thread structures, and generates companion images.
---

# Social Media Content Skill

## Research First (Required)
- Use the `task` tool with `subagent_type: "researcher"`
- Specify BOTH the topic AND where to save findings

## Output Structure
- LinkedIn: `linkedin/<slug>/post.md` + `image.png`
- Twitter/X: `tweets/<slug>/thread.md` + `image.png`

## LinkedIn Guidelines
- 1,300 character limit
- First line must hook attention
- 3-5 hashtags at the end
- Professional but personal tone

## Twitter/X Guidelines
- 280 character limit per tweet
- Threads: use 1/🧵 format
- Max 2 hashtags per tweet

## Social Image Best Practices
- Bold, simple compositions
- High contrast (stand out when scrolling)
- Single focal point
- Square or 4:5 ratio

## Content Types
- Announcement Posts: Lead with news, explain impact
- Insight Posts: Share one learning, make actionable
- Question Posts: Ask genuine question, provide your take first
"""

with open(skills_social / "SKILL.md", "w") as f:
    f.write(SKILL_SOCIAL_MEDIA)

print(f"社交媒体技能已创建: {skills_social / 'SKILL.md'}")

## 6. 自定义工具实现

   定义 Web 搜索和 AI 图片生成工具，供主智能体和子智能体使用。

In [ ]:
import os
from pathlib import Path
from typing import Literal

from langchain.tools import tool

EXAMPLE_DIR = Path.cwd() / "content-builder-agent"


@tool
def web_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news"] = "general",
) -> dict:
    """搜索 Web 获取最新信息。

    参数:
        query: 搜索查询（请尽量具体和详细）
        max_results: 返回结果数（默认: 5）
        topic: "general" 通用查询, "news" 新闻查询

    返回:
        包含标题、URL 和内容摘要的搜索结果。
    """
    try:
        from tavily import TavilyClient

        api_key = os.environ.get("TAVILY_API_KEY")
        if not api_key:
            return {"error": "TAVILY_API_KEY 未设置"}

        client = TavilyClient(api_key=api_key)
        return client.search(query, max_results=max_results, topic=topic)
    except Exception as e:
        return {"error": f"搜索失败: {e}"}


@tool
def generate_cover(prompt: str, slug: str) -> str:
    """为博客文章生成封面图片。

    参数:
        prompt: 要生成图片的详细描述
        slug: 博客文章的 slug，图片保存至 blogs/<slug>/hero.png
    """
    try:
        from google import genai

        client = genai.Client()
        response = client.models.generate_content(
            model="gemini-2.5-flash-image",
            contents=[prompt],
        )

        for part in response.parts:
            if part.inline_data is not None:
                image = part.as_image()
                output_path = EXAMPLE_DIR / "blogs" / slug / "hero.png"
                output_path.parent.mkdir(parents=True, exist_ok=True)
                image.save(str(output_path))
                return f"图片已保存至 {output_path}"

        return "未生成图片"
    except Exception as e:
        return f"错误: {e}"


@tool
def generate_social_image(prompt: str, platform: str, slug: str) -> str:
    """为社交媒体帖子生成配图。

    参数:
        prompt: 图片描述
        platform: "linkedin" 或 "tweets"
        slug: 帖子 slug，图片保存至 <platform>/<slug>/image.png
    """
    try:
        from google import genai

        client = genai.Client()
        response = client.models.generate_content(
            model="gemini-2.5-flash-image",
            contents=[prompt],
        )

        for part in response.parts:
            if part.inline_data is not None:
                image = part.as_image()
                output_path = EXAMPLE_DIR / platform / slug / "image.png"
                output_path.parent.mkdir(parents=True, exist_ok=True)
                image.save(str(output_path))
                return f"图片已保存至 {output_path}"

        return "未生成图片"
    except Exception as e:
        return f"错误: {e}"


print("所有自定义工具定义完成")

## 7. 加载子智能体配置

   从 YAML 文件中读取子智能体定义，并将工具名称解析为实际的函数对象。

In [ ]:
import yaml
from pathlib import Path


def load_subagents(config_path: Path) -> list:
    """从 YAML 加载子智能体定义并绑定工具。

    memory 和 skills 可以直接由 deep agents 自动加载，
    但子智能体需要此辅助函数来从外部 YAML 配置。
    """
    available_tools = {
        "web_search": web_search,
    }

    with open(config_path) as f:
        config = yaml.safe_load(f)

    subagents = []
    for name, spec in config.items():
        subagent = {
            "name": name,
            "description": spec["description"],
            "system_prompt": spec["system_prompt"],
        }
        if "model" in spec:
            subagent["model"] = spec["model"]
        if "tools" in spec:
            subagent["tools"] = [available_tools[t] for t in spec["tools"]]
        subagents.append(subagent)

    return subagents


# 测试加载
subagents = load_subagents(EXAMPLE_DIR / "subagents.yaml")
print(f"已加载 {len(subagents)} 个子智能体:")
for sa in subagents:
    print(f"  - {sa['name']}: {sa['description'][:50]}...")

## 8. 创建内容写作智能体

   整合所有组件：模型、记忆、技能、工具、子智能体和文件系统后端。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend


def create_content_writer():
    """创建由文件系统配置驱动的内容写作智能体"""
    return create_deep_agent(
        # 选择模型提供商
        # model="google_genai:gemini-3.5-flash",
        # model="openai:gpt-5.4",
        # model="anthropic:claude-sonnet-4-6",
        model="google_genai:gemini-3.5-flash",
        memory=["./AGENTS.md"],  # 品牌记忆文件
        skills=["./skills/"],    # 技能文件夹
        tools=[generate_cover, generate_social_image],  # 图片生成工具
        subagents=load_subagents(EXAMPLE_DIR / "subagents.yaml"),
        backend=FilesystemBackend(root_dir=EXAMPLE_DIR),
    )


# agent = create_content_writer()
print("内容写作智能体创建函数已定义")
print("使用 agent = create_content_writer() 实例化")

## 9. 运行内容生成任务

   向智能体提交内容创作请求，观察它如何研究、撰写并配图。

In [ ]:
# 示例：运行内容撰写任务
# 注意：需要有效的 API 密钥

# agent = create_content_writer()
# input_message = {
#     "role": "user",
#     "content": "Write a blog post about the rise of AI agents in enterprise software development"
# }

# for step in agent.stream(
#     {"messages": [input_message]},
#     stream_mode="updates",
# ):
#     for _, update in step.items():
#         if update and (messages := update.get("messages")):
#             for message in messages:
#                 message.pretty_print()

print("运行内容生成任务（取消注释以执行）")

## 10. 文件产出结构

    | 目录 | 内容 |
    |------|------|
    | `blogs/<slug>/post.md` | 博客文章正文 |
    | `blogs/<slug>/hero.png` | 博客封面图片 |
    | `linkedin/<slug>/post.md` | LinkedIn 帖子内容 |
    | `linkedin/<slug>/image.png` | LinkedIn 配图 |
    | `tweets/<slug>/thread.md` | Twitter/X 线程内容 |
    | `tweets/<slug>/image.png` | Twitter 配图 |
    | `research/<slug>.md` | 研究笔记 |

---

**参考链接**
- [LangChain Docs — Deep Agents Content Builder](https://docs.langchain.com/deep-agents/content-builder)
- [Deep Agents GitHub 示例 — content-builder-agent](https://github.com/langchain-ai/deep-agents)
- [Gemini Image Generation API](https://ai.google.dev)
- [Tavily Search API](https://tavily.com)